In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT  = Path(r"G:\FarmWise")
RAW_DIR       = PROJECT_ROOT / "Raw_Data"
PROCESSED_DIR = PROJECT_ROOT / "Processed_Data"
DOC_DIR       = PROJECT_ROOT / "Documentation" / "Data_Sources"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

power_path = RAW_DIR / "NASA_POWER" / "nasa_power_elzankalon_2015_2025.csv"
power_out  = PROCESSED_DIR / "NASA_POWER_Final.csv"
power_doc  = DOC_DIR / "NASA_POWER_Documentation.md"

print("Raw exists:", power_path.exists())

Raw exists: True


In [ ]:
with open(power_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

header_end  = next(i for i, l in enumerate(lines) if "-END HEADER-" in l)
header_text = "".join(lines[:header_end + 1])
print("Rows to skip:", header_end + 1)

power_raw = pd.read_csv(power_path, skiprows=header_end + 1)  
df = power_raw.copy()                                          

print(df.head())
print(df.tail())

Rows to skip: 15
   YEAR  DOY  ALLSKY_SFC_SW_DWN    T2M  T2M_MAX  T2M_MIN   RH2M  WS2M  \
0  2015    1              10.41  10.78    16.83     6.40  40.29  3.15   
1  2015    2              12.23  11.23    19.55     4.52  58.83  1.95   
2  2015    3              13.09  11.16    18.54     5.19  49.09  3.17   
3  2015    4              11.56  11.96    19.03     6.97  65.17  2.90   
4  2015    5              10.97  12.61    18.41     6.97  59.42  5.48   

   PRECTOTCORR  
0         0.00  
1         0.00  
2         0.19  
3         0.02  
4         0.01  
      YEAR  DOY  ALLSKY_SFC_SW_DWN    T2M  T2M_MAX  T2M_MIN   RH2M  WS2M  \
4013  2025  361              11.48  14.90    22.24     9.37  71.25  2.79   
4014  2025  362              11.57  14.26    20.45    10.38  68.57  3.04   
4015  2025  363              10.27  15.06    21.33    10.60  72.22  3.32   
4016  2025  364              12.07  15.34    21.16    10.57  62.37  1.61   
4017  2025  365              12.83  14.50    21.04     8.84  4

In [6]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)

print("\nMissing (NaN) per column:\n", df.isna().sum())
print("\nCount of -999 per column:\n", (df == -999).sum())

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("\nDescribe:\n", df.describe().T)

Shape: (4018, 9)

Columns: ['YEAR', 'DOY', 'ALLSKY_SFC_SW_DWN', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'WS2M', 'PRECTOTCORR']

Dtypes:
 YEAR                   int64
DOY                    int64
ALLSKY_SFC_SW_DWN    float64
T2M                  float64
T2M_MAX              float64
T2M_MIN              float64
RH2M                 float64
WS2M                 float64
PRECTOTCORR          float64
dtype: object

Missing (NaN) per column:
 YEAR                 0
DOY                  0
ALLSKY_SFC_SW_DWN    0
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS2M                 0
PRECTOTCORR          0
dtype: int64

Count of -999 per column:
 YEAR                 0
DOY                  0
ALLSKY_SFC_SW_DWN    0
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS2M                 0
PRECTOTCORR          0
dtype: int64

Describe:
                     count         mean         std      min        25%       50%   

In [7]:
df["date"] = pd.to_datetime(
    df["YEAR"].astype(str) + df["DOY"].astype(str).str.zfill(3),
    format="%Y%j"
)

expected = pd.date_range("2015-01-01", "2025-12-31", freq="D")

print("First date:", df["date"].min().date(), "| Last date:", df["date"].max().date())
print("Rows:", len(df), "| Expected days:", len(expected))
print("Duplicate dates:", df["date"].duplicated().sum())
print("Is sorted ascending:", df["date"].is_monotonic_increasing)

missing_dates = expected.difference(df["date"])
extra_dates   = pd.DatetimeIndex(df["date"]).difference(expected)
print("Missing dates:", len(missing_dates), list(missing_dates[:10]))
print("Unexpected dates:", len(extra_dates))

print("Years with DOY=366:", sorted(df.loc[df["DOY"] == 366, "YEAR"].unique()))

First date: 2015-01-01 | Last date: 2025-12-31
Rows: 4018 | Expected days: 4018
Duplicate dates: 0
Is sorted ascending: True
Missing dates: 0 []
Unexpected dates: 0
Years with DOY=366: [np.int64(2016), np.int64(2020), np.int64(2024)]


In [ ]:
# 1) Physical plausibility
rules = {
    "T2M":               (-10, 50),
    "T2M_MAX":           (-5, 55),
    "T2M_MIN":           (-15, 40),
    "RH2M":              (0, 100),
    "WS2M":              (0, 30),
    "ALLSKY_SFC_SW_DWN": (0, 40),
    "PRECTOTCORR":       (0, 500),
}
print("=== Physical range violations ===")
for col, (lo, hi) in rules.items():
    bad = df[(df[col] < lo) | (df[col] > hi)]
    print(f"{col:20s} range [{lo}, {hi}] -> violations: {len(bad)}")

# 2) Logical consistency
print("\n=== Logical checks ===")
print("T2M_MAX < T2M :", (df["T2M_MAX"] < df["T2M"]).sum())
print("T2M < T2M_MIN :", (df["T2M"] < df["T2M_MIN"]).sum())
print("T2M_MAX < T2M_MIN:", (df["T2M_MAX"] < df["T2M_MIN"]).sum())

# 3) Outliers (flag only) - monthly IQR x 3
print("\n=== Outliers (monthly IQR x 3) - NOT deleted ===")
month = df["date"].dt.month
outlier_report = {}
for col in ["T2M", "T2M_MAX", "T2M_MIN", "RH2M", "WS2M", "ALLSKY_SFC_SW_DWN"]:
    q1 = df.groupby(month)[col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(month)[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    flag = (df[col] < q1 - 3 * iqr) | (df[col] > q3 + 3 * iqr)
    outlier_report[col] = df.loc[flag, ["date", col]]
    print(f"{col:20s} flagged: {flag.sum()}")


print("\nTop 10 rainiest days (PRECTOTCORR, mm/day):")
print(df.nlargest(10, "PRECTOTCORR")[["date", "PRECTOTCORR"]])
print("\nRain days (>=1 mm):", (df["PRECTOTCORR"] >= 1).sum(),
      "| Exact zero days:", (df["PRECTOTCORR"] == 0).sum())

for col, rep in outlier_report.items():
    if len(rep):
        print(f"\n--- {col} ---")
        print(rep.head(10).to_string(index=False))

=== Physical range violations ===
T2M                  range [-10, 50] -> violations: 0
T2M_MAX              range [-5, 55] -> violations: 0
T2M_MIN              range [-15, 40] -> violations: 0
RH2M                 range [0, 100] -> violations: 0
WS2M                 range [0, 30] -> violations: 0
ALLSKY_SFC_SW_DWN    range [0, 40] -> violations: 0
PRECTOTCORR          range [0, 500] -> violations: 0

=== Logical checks ===
T2M_MAX < T2M : 0
T2M < T2M_MIN : 0
T2M_MAX < T2M_MIN: 0

=== Outliers (monthly IQR x 3) - NOT deleted ===
T2M                  flagged: 0
T2M_MAX              flagged: 0
T2M_MIN              flagged: 0
RH2M                 flagged: 5
WS2M                 flagged: 3
ALLSKY_SFC_SW_DWN    flagged: 35

Top 10 rainiest days (PRECTOTCORR, mm/day):
           date  PRECTOTCORR
1897 2020-03-12        64.01
1941 2020-04-25        35.62
1940 2020-04-24        33.97
831  2017-04-11        31.48
691  2016-11-22        28.39
699  2016-11-30        26.33
1938 2020-04-22        

In [9]:
value_cols = ["ALLSKY_SFC_SW_DWN", "T2M", "T2M_MAX", "T2M_MIN", "RH2M", "WS2M", "PRECTOTCORR"]
df[value_cols] = df[value_cols].replace(-999, np.nan)


rename_map = {
    "ALLSKY_SFC_SW_DWN": "solar_rad_mj_m2_day",
    "T2M":               "t2m_c",
    "T2M_MAX":           "t2m_max_c",
    "T2M_MIN":           "t2m_min_c",
    "RH2M":              "rh2m_pct",
    "WS2M":              "ws2m_ms",
    "PRECTOTCORR":       "precip_mm_day",
}
power_clean = df.rename(columns=rename_map)

power_clean = power_clean[["date", "t2m_c", "t2m_max_c", "t2m_min_c",
                           "precip_mm_day", "rh2m_pct", "ws2m_ms", "solar_rad_mj_m2_day"]]


power_clean["date"] = pd.to_datetime(power_clean["date"]).dt.normalize()
power_clean = power_clean.sort_values("date").drop_duplicates(subset="date", keep="first").reset_index(drop=True)
for c in power_clean.columns[1:]:
    power_clean[c] = power_clean[c].astype("float64")

print("Shape after cleaning:", power_clean.shape)
print("Missing per column:\n", power_clean.isna().sum())
print(power_clean.dtypes)
print(power_clean.head())

Shape after cleaning: (4018, 8)
Missing per column:
 date                   0
t2m_c                  0
t2m_max_c              0
t2m_min_c              0
precip_mm_day          0
rh2m_pct               0
ws2m_ms                0
solar_rad_mj_m2_day    0
dtype: int64
date                   datetime64[us]
t2m_c                         float64
t2m_max_c                     float64
t2m_min_c                     float64
precip_mm_day                 float64
rh2m_pct                      float64
ws2m_ms                       float64
solar_rad_mj_m2_day           float64
dtype: object
        date  t2m_c  t2m_max_c  t2m_min_c  precip_mm_day  rh2m_pct  ws2m_ms  solar_rad_mj_m2_day
0 2015-01-01  10.78      16.83       6.40           0.00     40.29     3.15                10.41
1 2015-01-02  11.23      19.55       4.52           0.00     58.83     1.95                12.23
2 2015-01-03  11.16      18.54       5.19           0.19     49.09     3.17                13.09
3 2015-01-04  11.96      19.

In [10]:
# Check original NASA POWER header and units
print("=== NASA POWER HEADER ===")

for line in header_text.splitlines():
    if any(key in line for key in [
        "ALLSKY_SFC_SW_DWN",
        "T2M",
        "PRECTOTCORR"
    ]):
        print(line)

# Inspect flagged outliers
print("\n=== RH2M OUTLIERS ===")
print(outlier_report["RH2M"].to_string(index=False))

print("\n=== WS2M OUTLIERS ===")
print(outlier_report["WS2M"].to_string(index=False))

print("\n=== SOLAR RADIATION OUTLIERS ===")
print(outlier_report["ALLSKY_SFC_SW_DWN"].to_string(index=False))

# Inspect highest precipitation values
print("\n=== TOP 10 RAINIEST DAYS ===")
print(
    df.nlargest(10, "PRECTOTCORR")[
        ["date", "PRECTOTCORR"]
    ].to_string(index=False)
)

=== NASA POWER HEADER ===
ALLSKY_SFC_SW_DWN     CERES SYN1deg All Sky Surface Shortwave Downward Irradiance (MJ/m^2/day) 
T2M                   MERRA-2 Temperature at 2 Meters (C) 
T2M_MAX               MERRA-2 Temperature at 2 Meters Maximum (C) 
T2M_MIN               MERRA-2 Temperature at 2 Meters Minimum (C) 
PRECTOTCORR           MERRA-2 Precipitation Corrected (mm/day) 

=== RH2M OUTLIERS ===
      date  RH2M
2016-06-03 17.59
2017-10-25 34.28
2023-08-05 27.76
2023-08-06 28.29
2023-09-07 29.30

=== WS2M OUTLIERS ===
      date  WS2M
2015-02-10  6.25
2015-02-11  6.49
2024-10-23  4.87

=== SOLAR RADIATION OUTLIERS ===
      date  ALLSKY_SFC_SW_DWN
2015-06-06              20.65
2015-06-07              22.61
2015-09-09              14.71
2015-10-25               8.09
2015-11-15               6.68
2017-11-21               6.34
2018-05-04              11.55
2018-05-05              15.53
2018-05-06              10.53
2019-04-15               8.24
2019-06-04              22.99
2019-06-05 

In [11]:
from pathlib import Path

power_out = Path(power_out)

power_out.parent.mkdir(parents=True, exist_ok=True)

power_clean.to_csv(
    power_out,
    index=False,
    date_format="%Y-%m-%d"
)

print("NASA POWER file saved successfully!")
print("Output path:", power_out)
print("Shape:", power_clean.shape)

NASA POWER file saved successfully!
Output path: G:\FarmWise\Processed_Data\NASA_POWER_Final.csv
Shape: (4018, 8)


In [12]:
import pandas as pd
import numpy as np

# Reload saved file
chk = pd.read_csv(power_out, parse_dates=["date"])

print("=== FINAL VALIDATION ===")

# 1. Shape
print("\nShape:", chk.shape)
print("Expected rows:", 4018)
print("Actual rows:", len(chk))

# 2. Missing values
print("\n=== Missing Values ===")
print(chk.isna().sum())

# 3. Duplicate dates
print("\nDuplicate dates:", chk["date"].duplicated().sum())

# 4. Date range and continuity
expected_dates = pd.date_range(
    start="2015-01-01",
    end="2025-12-31",
    freq="D"
)

actual_dates = pd.DatetimeIndex(chk["date"])

missing_dates = expected_dates.difference(actual_dates)
unexpected_dates = actual_dates.difference(expected_dates)

print("\nDate range:", chk["date"].min(), "to", chk["date"].max())
print("Missing dates:", len(missing_dates))
print("Unexpected dates:", len(unexpected_dates))
print("Dates sorted:", chk["date"].is_monotonic_increasing)

# 5. Physical range validation
ranges = {
    "t2m_c": (-10, 50),
    "t2m_max_c": (-5, 55),
    "t2m_min_c": (-15, 40),
    "rh2m_pct": (0, 100),
    "ws2m_ms": (0, 30),
    "solar_rad_mj_m2_day": (0, 40),
    "precip_mm_day": (0, 500)
}

print("\n=== Physical Range Checks ===")

range_results = {}

for col, (lower, upper) in ranges.items():
    violations = (
        (chk[col] < lower) |
        (chk[col] > upper)
    ).sum()

    range_results[col] = violations == 0

    print(f"{col}: {violations} violations")

# 6. Logical consistency
print("\n=== Logical Checks ===")

max_below_avg = (chk["t2m_max_c"] < chk["t2m_c"]).sum()
avg_below_min = (chk["t2m_c"] < chk["t2m_min_c"]).sum()
max_below_min = (chk["t2m_max_c"] < chk["t2m_min_c"]).sum()

print("T2M_MAX < T2M:", max_below_avg)
print("T2M < T2M_MIN:", avg_below_min)
print("T2M_MAX < T2M_MIN:", max_below_min)

# 7. Overall result
checks = {
    "Correct row count": len(chk) == 4018,
    "No missing values": chk.isna().sum().sum() == 0,
    "No duplicate dates": chk["date"].duplicated().sum() == 0,
    "No missing dates": len(missing_dates) == 0,
    "No unexpected dates": len(unexpected_dates) == 0,
    "Dates sorted": chk["date"].is_monotonic_increasing,
    "Physical ranges valid": all(range_results.values()),
    "Logical checks valid": (
        max_below_avg == 0 and
        avg_below_min == 0 and
        max_below_min == 0
    )
}

print("\n=== VALIDATION SUMMARY ===")

for check, result in checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print(
    "\nFINAL RESULT:",
    "ALL CHECKS PASSED" if all(checks.values())
    else "SOME CHECKS FAILED"
)

assert all(checks.values()), "Validation failed. Review the checks above."

=== FINAL VALIDATION ===

Shape: (4018, 8)
Expected rows: 4018
Actual rows: 4018

=== Missing Values ===
date                   0
t2m_c                  0
t2m_max_c              0
t2m_min_c              0
precip_mm_day          0
rh2m_pct               0
ws2m_ms                0
solar_rad_mj_m2_day    0
dtype: int64

Duplicate dates: 0

Date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00
Missing dates: 0
Unexpected dates: 0
Dates sorted: True

=== Physical Range Checks ===
t2m_c: 0 violations
t2m_max_c: 0 violations
t2m_min_c: 0 violations
rh2m_pct: 0 violations
ws2m_ms: 0 violations
solar_rad_mj_m2_day: 0 violations
precip_mm_day: 0 violations

=== Logical Checks ===
T2M_MAX < T2M: 0
T2M < T2M_MIN: 0
T2M_MAX < T2M_MIN: 0

=== VALIDATION SUMMARY ===
Correct row count: PASS
No missing values: PASS
No duplicate dates: PASS
No missing dates: PASS
No unexpected dates: PASS
Dates sorted: PASS
Physical ranges valid: PASS
Logical checks valid: PASS

FINAL RESULT: ALL CHECKS PASSED


In [13]:
from pathlib import Path
from datetime import datetime

# Documentation path
docs_folder = Path(r"G:\FarmWise\Documentation\Data_Sources")
docs_folder.mkdir(parents=True, exist_ok=True)

doc_path = docs_folder / "NASA_POWER_Documentation.md"

documentation = f"""# NASA POWER Daily Weather Data

## 1. Data Source

- **Source:** NASA POWER
- **Product:** Daily Point Meteorological and Solar Data
- **Website:** https://power.larc.nasa.gov/
- **Meteorological data source:** MERRA-2
- **Solar radiation data source:** CERES SYN1deg

## 2. Study Area

- **Location:** El-Zankalon, Sharqia, Egypt
- **Study period:** 2015-01-01 to 2025-12-31

## 3. Raw Data

- **Raw file:** nasa_power_elzankalon_2015_2025.csv
- **Raw folder:** G:/FarmWise/Raw_Data/NASA_POWER/
- **Original records:** 4018
- **Original columns:** 9

The raw file was preserved without modification.

## 4. Variables

| Original Variable | Processed Column | Unit | Description |
|---|---|---|---|
| YEAR + DOY | date | YYYY-MM-DD | Observation date |
| T2M | t2m_c | °C | Temperature at 2 meters |
| T2M_MAX | t2m_max_c | °C | Maximum temperature |
| T2M_MIN | t2m_min_c | °C | Minimum temperature |
| PRECTOTCORR | precip_mm_day | mm/day | Corrected precipitation |
| RH2M | rh2m_pct | % | Relative humidity at 2 meters |
| WS2M | ws2m_ms | m/s | Wind speed at 2 meters |
| ALLSKY_SFC_SW_DWN | solar_rad_mj_m2_day | MJ/m²/day | All-sky surface shortwave downward irradiance |

## 5. Data Cleaning

The following operations were performed:

1. Skipped the metadata/header lines before reading the tabular data.
2. Combined YEAR and DOY to create a proper date column.
3. Checked for missing values and invalid fill values (-999).
4. Renamed columns to descriptive names.
5. Sorted records chronologically.
6. Checked for duplicate dates.
7. Converted measurement columns to numeric data types.
8. Preserved the original raw dataset.

No outlier values were automatically deleted.

## 6. Outlier Analysis

Monthly IQR analysis using a multiplier of 3 was applied to selected variables.

| Variable | Flagged Outliers |
|---|---:|
| T2M | 0 |
| T2M_MAX | 0 |
| T2M_MIN | 0 |
| RH2M | 5 |
| WS2M | 3 |
| Solar Radiation | 35 |

A total of 43 observations were flagged.

These observations were retained because statistical outlier detection alone does not prove that a measurement is invalid.

## 7. Validation

Final validation was performed on the saved processed CSV.

- Expected daily records: 4018
- Actual daily records: {len(chk)}
- Date range: {chk['date'].min().date()} to {chk['date'].max().date()}
- Missing values: {int(chk.isna().sum().sum())}
- Duplicate dates: {int(chk['date'].duplicated().sum())}
- Missing dates: {len(missing_dates)}
- Unexpected dates: {len(unexpected_dates)}

### Physical and Logical Checks

Physical ranges and temperature relationships were checked.

Validation results:

{chr(10).join(f"- {name}: {'PASS' if result else 'FAIL'}" for name, result in checks.items())}

## 8. Processed Data

- **Output file:** {power_out.name}
- **Output folder:** G:/FarmWise/Processed_Data/NASA_POWER/
- **Final shape:** {chk.shape}
- **Status:** {'All validation checks passed' if all(checks.values()) else 'Validation requires review'}

## 9. Limitations and Notes

- NASA POWER provides gridded/reanalysis-based and satellite-derived estimates, not direct measurements from every farm field.
- Outliers were flagged for review but not removed automatically.
- The dataset ends on 2025-12-31. Other project datasets may have different temporal coverage.
- Further spatial and temporal alignment may be required before combining this dataset with MODIS NDVI and GPM precipitation.

---
Documentation generated: {datetime.now().strftime('%Y-%m-%d')}
"""

with open(doc_path, "w", encoding="utf-8") as file:
    file.write(documentation)

print("Documentation created successfully!")
print("Documentation path:", doc_path)

Documentation created successfully!
Documentation path: G:\FarmWise\Documentation\Data_Sources\NASA_POWER_Documentation.md
